# Drone imagery (TERN STAC API)
Search imagery items, load with OpenDataCube integration, and preview RGB.

### Data used:

[Dronescape UAS multispectral collection](https://stac-api.tern.org.au/stac-browser/collections/uas__dronescape_multispec)

[Specific Dataset used](https://stac-api.tern.org.au/stac-browser/collections/uas__dronescape_multispec/items/20241001_SAAGAW0009_multispec_ortho_02_cog)



## Before you run
- Update placeholder values (`COLLECTION_ID`, dates, bounds, point coordinates) to match your data.
- Ensure auth is configured for protected assets (for example `.netrc` and/or GDAL config).
- Install optional dependencies required by this notebook's workflow (`odc-stac`, `rioxarray`, `geopandas`, plotting extras).
- Run cells from top to bottom so variables are initialized in order.

### API Key

[Create API Key](https://ternaus.atlassian.net/wiki/spaces/TERNSup/pages/2353496065/Creating+API+Keys)

[Use API Key](https://ternaus.atlassian.net/wiki/spaces/TERNSup/pages/3355246599/Using+API+Keys+to+Access+TERN+Data+Services#Create-your-API-Key)

In [1]:
from tern_stac import TernStacClient, load_items_odc, preview_raster
import geopandas as gpd

## Define some variables

These are all query parameters for STAC API to retrieve items of interest.

In [3]:
# Fill in from your catalog values
# Find the collection ID, item ID, and asset key from stac api browser
# https://stac-api.tern.org.au/stac-browser/?.language=en
COLLECTION_ID = "uas__dronescape_multispec"
# COLLECTION_ID = "uas__dronescape_rgb"
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
# false-color vegetation composite - Color Infrared (CIR) composite
# Vegetation typically appears red/pink, while water tends to be dark and soil/bare ground tends toward brown/tan.
# R = B10 (842 nm)   ← NIR
# G = B6 (668 nm)    ← Red
# B = B4 (560 nm)    ← Green
BANDS = ["b10", "b6", "b4"]
REGION_BOUNDS = (135.6242307, -30.6713657, 135.6290876, -30.6668747)  # (minx, miny, maxx, maxy)
REGION_BOUNDS_CRS = "EPSG:4326"
# or just get the item ID and asset key from the stac api browser

## Plot the Region on an interactive map

This is to get an idea about where the bounding box we are looking exactly is.

(You may need to use jupyterlabs nbviewer to see the map rendered. https://nbviewer.org/github/ternaustralia/TERN-Data-Skills/blob/master/UQRandI2026/03_Drone_Imagery.ipynb)

In [20]:
# plot the bounds
import folium

minx, miny, maxx, maxy = REGION_BOUNDS
centre = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(
    location=centre,
    zoom_start=11,
    # tiles="Esri.WorldImagery",
    tiles="OpenStreetMap",
)

folium.Rectangle(
    bounds=[[miny, minx], [maxy, maxx]],
    color="red",
    weight=3,
    fill=False
).add_to(m)

m

## Find the items for give time range and region in the STAC Catalog

In [6]:
client = TernStacClient()
search = client.search(
    collections=[COLLECTION_ID],
    datetime=f"{START_DATE}/{END_DATE}",
    bbox=[REGION_BOUNDS[0], REGION_BOUNDS[1], REGION_BOUNDS[2], REGION_BOUNDS[3]],
)
items = list(search.items())
print(len(items), "Item(s) found")

1 Item(s) found


## Inspect Item

In [7]:
items[0]

<Item id=20241001_SAAGAW0009_multispec_ortho_02_cog>

## Load bands of interest at reduced resolution (1m)

This uses OpenDataCube integration which allows filtering by bands and re-scale resolution

In [8]:
ds = load_items_odc(
    items,
    bands=BANDS,
    crs="utm",
    groupby="solar_day",
    resolution=1,
    chunks={},
)
ds

<xarray.Dataset> Size: 3MB
Dimensions:      (y: 497, x: 464, time: 1)
Coordinates:
  * y            (y) float64 4kB 6.607e+06 6.607e+06 ... 6.607e+06 6.607e+06
  * x            (x) float64 4kB 5.598e+05 5.598e+05 ... 5.603e+05 5.603e+05
  * time         (time) datetime64[ns] 8B 2024-10-01
    spatial_ref  int32 4B 32753
Data variables:
    b10          (time, y, x) float32 922kB dask.array<chunksize=(1, 497, 464), meta=np.ndarray>
    b6           (time, y, x) float32 922kB dask.array<chunksize=(1, 497, 464), meta=np.ndarray>
    b4           (time, y, x) float32 922kB dask.array<chunksize=(1, 497, 464), meta=np.ndarray>

## Plot the false-color vegetation composite - Color Infrared (CIR) composite

In [10]:
# Vegetation typically appears red/pink, while water tends to be dark and soil/bare ground tends toward brown/tan.

# Plot is rendered as image to display in nbviewer.
import os
import matplotlib.pyplot as plt
# ensure target directory exists
os.makedirs("images", exist_ok=True)

with plt.ioff():
    preview_raster(
        ds,
        rgb_bands=BANDS,
        time_index=-1,
        title="Imagery ROI RGB (last timestep)",
        save_path="images/03_Drone_Imagery.png"
    )

![False Color CIR plot](images/03_Drone_Imagery.png)
